# Data Cleaning and Preparation Project
## Global Superstore Dataset

**SkillCraft Technology - Data Analytics Internship Task 2**

---

### Project Objectives
1. Load and inspect the dataset
2. Handle missing values
3. Remove duplicates
4. Standardize column names
5. Convert data types appropriately
6. Remove unnecessary whitespace
7. Detect outliers
8. Export cleaned dataset

---

## 1. Import Required Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# System utilities
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

print("✓ Libraries imported successfully!")

## 2. Load the Dataset

In [ ]:
# Load the raw dataset
df = pd.read_csv('../Dataset/raw/Global_Superstore.csv')

# Create a copy for comparison later
df_original = df.copy()

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

## 3. Initial Data Inspection

Let's examine the structure and content of our dataset.

In [ ]:
# Display first few rows
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Dataset information
print("Dataset Information:")
df.info()

In [ ]:
# Column names and data types
print("\nColumn Names and Data Types:")
for i, (col, dtype) in enumerate(zip(df.columns, df.dtypes), 1):
    print(f"{i:2d}. {col:20s} : {dtype}")

In [ ]:
# Statistical summary
print("Statistical Summary of Numerical Columns:")
df.describe()

## 4. Missing Value Analysis

Identifying and handling missing values is crucial for data quality.

In [ ]:
# Count missing values
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Percentage': (df.isnull().sum().values / len(df) * 100).round(2)
})

missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values(
    'Missing_Count', ascending=False
)

print("Missing Values Summary:")
if len(missing_data) > 0:
    print(missing_data.to_string(index=False))
else:
    print("✓ No missing values found!")

In [ ]:
# Visualize missing values
if len(missing_data) > 0:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=missing_data, x='Column', y='Missing_Percentage', palette='viridis')
    plt.title('Missing Values by Column', fontsize=14, fontweight='bold')
    plt.xlabel('Column Name', fontsize=12)
    plt.ylabel('Missing Percentage (%)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

### Handle Missing Values

Strategy:
- **Numerical columns**: Fill with median
- **Categorical columns**: Fill with mode or 'Unknown'
- **High missingness (>50%)**: Drop column

In [ ]:
# Handle missing values
for col in df.columns:
    missing_count = df[col].isnull().sum()
    
    if missing_count > 0:
        missing_pct = (missing_count / len(df)) * 100
        
        if missing_pct > 50:
            df.drop(columns=[col], inplace=True)
            print(f"Dropped '{col}' (>50% missing)")
            
        elif df[col].dtype in ['int64', 'float64']:
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            print(f"Filled '{col}' with median: {median_val}")
            
        elif df[col].dtype == 'object':
            if df[col].mode().empty:
                df[col].fillna('Unknown', inplace=True)
                print(f"Filled '{col}' with 'Unknown'")
            else:
                mode_val = df[col].mode()[0]
                df[col].fillna(mode_val, inplace=True)
                print(f"Filled '{col}' with mode: {mode_val}")

print(f"\n✓ Missing values handled! Total missing now: {df.isnull().sum().sum()}")

## 5. Duplicate Detection and Removal

In [ ]:
# Check for duplicates
duplicates_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates_count}")

if duplicates_count > 0:
    # Show some duplicate examples
    print("\nExample duplicate rows:")
    print(df[df.duplicated()].head())
    
    # Remove duplicates
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"\n✓ Removed {duplicates_count} duplicate rows")
    print(f"New shape: {df.shape}")
else:
    print("✓ No duplicates found!")

## 6. Column Name Standardization

Convert column names to lowercase with underscores for consistency.

In [ ]:
print("Original column names:")
print(list(df.columns))

# Standardize column names
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('-', '_')

print("\nStandardized column names:")
print(list(df.columns))
print("\n✓ Column names standardized!")

## 7. Data Type Conversion

Ensure each column has the appropriate data type.

In [ ]:
print("Data types before conversion:")
print(df.dtypes)

# Convert postal_code to string (it's an identifier)
if 'postal_code' in df.columns:
    df['postal_code'] = df['postal_code'].astype(str)
    print("\n✓ Converted 'postal_code' to string")

# Convert quantity to integer
if 'quantity' in df.columns:
    df['quantity'] = df['quantity'].astype(int)
    print("✓ Converted 'quantity' to integer")

# Ensure numeric columns are float
numeric_cols = ['sales', 'discount', 'profit']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f"✓ Ensured '{col}' is numeric (float)")

print("\nData types after conversion:")
print(df.dtypes)

## 8. Remove Whitespace

Clean text columns by removing leading and trailing spaces.

In [ ]:
# Get text columns
text_columns = df.select_dtypes(include=['object']).columns
print(f"Processing {len(text_columns)} text columns...\n")

# Remove whitespace
for col in text_columns:
    df[col] = df[col].astype(str).str.strip()
    print(f"✓ Cleaned '{col}'")

print("\n✓ Whitespace removed from all text columns!")

## 9. Outlier Detection (IQR Method)

Identify outliers using the Interquartile Range method.

**Formula:**
- IQR = Q3 - Q1
- Lower Bound = Q1 - 1.5 × IQR
- Upper Bound = Q3 + 1.5 × IQR

In [ ]:
# Get numerical columns
numeric_columns = df.select_dtypes(include=[np.number]).columns

outlier_summary = []

for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_count = len(outliers)
    outlier_pct = (outlier_count / len(df)) * 100
    
    if outlier_count > 0:
        outlier_summary.append({
            'Column': col,
            'Outlier_Count': outlier_count,
            'Percentage': round(outlier_pct, 2),
            'Lower_Bound': round(lower_bound, 2),
            'Upper_Bound': round(upper_bound, 2),
            'Min_Outlier': round(outliers[col].min(), 2),
            'Max_Outlier': round(outliers[col].max(), 2)
        })

if outlier_summary:
    outlier_df = pd.DataFrame(outlier_summary)
    print("Outlier Summary:")
    print(outlier_df.to_string(index=False))
    print("\nNOTE: Outliers detected but NOT removed (may represent legitimate values)")
else:
    print("✓ No outliers detected!")

In [ ]:
# Visualize outliers with boxplots
if outlier_summary and len(outlier_summary) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Outlier Detection - Box Plots', fontsize=16, fontweight='bold')
    
    plot_cols = [item['Column'] for item in outlier_summary[:4]]
    
    for idx, col in enumerate(plot_cols):
        row = idx // 2
        col_idx = idx % 2
        
        axes[row, col_idx].boxplot(df[col].dropna(), vert=True)
        axes[row, col_idx].set_title(f'{col}', fontsize=12, fontweight='bold')
        axes[row, col_idx].set_ylabel('Value', fontsize=10)
        axes[row, col_idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 10. Before vs After Comparison

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Rows', 'Columns', 'Missing Values', 'Duplicates'],
    'Before Cleaning': [
        df_original.shape[0],
        df_original.shape[1],
        df_original.isnull().sum().sum(),
        df_original.duplicated().sum()
    ],
    'After Cleaning': [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum()
    ]
})

print("\n" + "="*60)
print("BEFORE vs AFTER COMPARISON")
print("="*60)
print(comparison.to_string(index=False))
print("="*60)

## 11. Final Dataset Preview

In [ ]:
print("Cleaned Dataset - First 10 rows:")
df.head(10)

In [ ]:
print("\nCleaned Dataset Info:")
df.info()

In [ ]:
print("\nCleaned Dataset Statistics:")
df.describe()

## 12. Export Cleaned Dataset

In [ ]:
# Create output directory if it doesn't exist
output_dir = Path('../Dataset/cleaned')
output_dir.mkdir(parents=True, exist_ok=True)

# Save cleaned dataset
output_path = '../Dataset/cleaned/Global_Superstore_Cleaned.csv'
df.to_csv(output_path, index=False)

print(f"✓ Cleaned dataset saved to: {output_path}")
print(f"✓ Shape: {df.shape[0]} rows × {df.shape[1]} columns")

## 13. Summary and Conclusions

### Data Cleaning Steps Completed:

1. ✓ **Data Loading**: Successfully loaded Global Superstore dataset
2. ✓ **Missing Values**: Identified and handled using appropriate imputation strategies
3. ✓ **Duplicates**: Detected and removed duplicate rows
4. ✓ **Column Names**: Standardized to lowercase with underscores
5. ✓ **Data Types**: Converted to appropriate types for analysis
6. ✓ **Whitespace**: Removed from all text columns
7. ✓ **Outliers**: Detected using IQR method (retained for business value)
8. ✓ **Export**: Saved cleaned dataset for downstream analysis

### Key Insights:

- The dataset is now **clean** and **analysis-ready**
- All missing values have been handled appropriately
- Data types are consistent and correct
- Outliers have been identified but retained (may represent bulk orders, high discounts, etc.)
- The dataset maintains high data quality and integrity

### Next Steps:

1. Perform exploratory data analysis (EDA)
2. Create visualizations to understand patterns
3. Build predictive models
4. Generate business insights

---

**Project completed successfully! 🎉**